# 21 · WGFMU IV sweep · L1 空连 (real B1500, no DUT)

> ⚠️ **执行前必读** ⚠️
>
> 此 notebook 真连 B1500，但**不接器件**，验证整条链路通。
>
> **前置确认**：
> - [ ] B1500 已开机、自检通过
> - [ ] Keysight IO Libraries (VISA) + NI-488.2 GPIB driver 装好
> - [ ] Connection Expert 能看到 B1500 (`*IDN?` 成功)
> - [ ] **Keysight B1530A Instrument Library 64-bit 已装** (dll 通常在 `C:\Windows\System32\wgfmu.dll`)
> - [ ] WGFMU 通道 **接了 RSU** (没接 RSU 的通道无法做 FASTIV)
> - [ ] 探针抬起或断开，没有 DUT 接入

**通过标准**：
- 无 dll 错
- VISA 自动找到 B1500
- 通道自动探测 + FASTIV 可用
- `complete == total`
- `qc.status == "ok"`
- 电流读数 **包含 RSU + 电缆 baseline (~1 µA 量级是正常的)**，不是漏电

测试人：**yhzang**


In [ ]:
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fefetlab.measurements.wgfmu import (
    ensure_wgfmu_dll_path,
    autodetect_visa_addr,
    autodetect_wgfmu_chan,
    RealWgfmuBackend,
    WgfmuIVSweepRunner,
    WgfmuIVSweepConfig,
)
from fefetlab.measurements.wgfmu.pulse_builder import linear_voltage_segments

print("Python:", sys.version.split()[0])
print("project root:", ROOT)


## 1. 自动找 wgfmu.dll + VISA + WGFMU 通道


In [ ]:
# 1.1 找 dll (System32 优先, 找不到就教用户装 Library)
dll = ensure_wgfmu_dll_path()
print(f"✅ wgfmu.dll: {dll}")

# 1.2 找 B1500 VISA 地址 (扫所有 GPIB 资源)
VISA_ADDR = autodetect_visa_addr("B1500")
print(f"✅ B1500 VISA addr: {VISA_ADDR}")

# 1.3 加载 backend, 拿通道列表
backend = RealWgfmuBackend()
backend.load()
backend.open_session(VISA_ADDR)
backend.set_timeout(30.0)

channel_ids = backend.get_channel_ids()
print(f"detected WGFMU channels: {channel_ids}")

# 1.4 选第一个接了 RSU 的通道作为默认 (yhzang 本机 201/202/301 接了 RSU, 302 没接)
CHAN_ID = autodetect_wgfmu_chan(backend, prefer=201 if 201 in channel_ids else None)
print(f"✅ using CHAN_ID = {CHAN_ID}")

# 拿完通道立刻 close, 避免和 runner.run 内部 open_session 冲突
backend.close_session()


## 2. 最小 IV sweep · 低压·1MA 量程 (容纳 RSU baseline ~1µA)

参数选择：
- `v_stop = -0.5V`：远低于任何 FeFET 阈值，安全
- `n_points = 5`：最少必要
- `current_range = "1MA"`：量程足够大，容纳 RSU+3m HRSU 电缆约 1µA baseline
- `t_high = 5e-6`：足够采到 20 个点


In [ ]:
segments = linear_voltage_segments(
    v_start=0.0,
    v_stop=-0.5,
    n_points=5,
    t_rise_s=1e-6,
    t_high_s=5e-6,
    t_fall_s=1e-6,
    t_base_s=2e-6,
    measure_points=20,
    measure_average_s=100e-9,
)

cfg = WgfmuIVSweepConfig(
    label="L1_aircheck_ivsweep",
    chan_id=CHAN_ID,
    v_init=0.0,
    v_base=0.0,
    operation_mode="FASTIV",
    force_voltage_range="AUTO",
    measure_mode="CURRENT",
    measure_current_range="1MA",   # 改: 容纳 RSU baseline (之前 1UA 会饱和)
    treat_warning_as_error=False,
    timeout_s=30.0,
    notes="L1 空连验证, 探针抬起, 预期电流量级 = RSU+电缆 baseline ~1µA",
)

print("segments:", len(segments), "expected pulses: 5")
print("cfg.chan_id:", cfg.chan_id)
print("cfg.measure_current_range:", cfg.measure_current_range)


In [ ]:
# === 真正调用 B1500 ===
runner = WgfmuIVSweepRunner(backend=backend)
result = runner.run(resource=VISA_ADDR, segments=segments, cfg=cfg)

print("✅ run() returned")
print(f"complete / total : {result.complete} / {result.total}")
print(f"issues           : {result.issues}")


## 3. QC + 数据


In [ ]:
print("=== qc_df ===")
print(result.qc_df.to_string(index=False))
print()
print("=== iv_df (per-pulse summary) ===")
print(result.iv_df.to_string(index=False))
print(f"\ntotal samples: {len(result.samples_df)}")

err = result.meta.get("error_summary", "")
wrn = result.meta.get("warning_summary", "")
print("\nerror_summary  :", err if err else "(empty)")
print("warning_summary:", wrn if wrn else "(empty)")
print(f"execution_time_s: {result.meta.get('execution_time_s'):.2f}")


## 4. 波形 + 电流图


In [ ]:
samples = result.samples_df
value_col = "current_A" if "current_A" in samples.columns else samples.columns[-1]

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
t_wf, v_wf = result.plan.waveform_samples(dt_s=5e-8)
axes[0].plot(t_wf*1e6, v_wf, color="#1f77b4", lw=1.0)
axes[0].set_ylabel("voltage (V)")
axes[0].set_title(f"L1 aircheck · IV sweep · {result.iv_df.shape[0]} pulses · CH{CHAN_ID}")
axes[0].grid(alpha=0.3)

axes[1].plot(samples["time_s"]*1e6, samples[value_col]*1e9, "o-",
             ms=3, lw=0.8, color="#d62728")
axes[1].set_xlabel("time (us)")
axes[1].set_ylabel(f"{value_col} (nA)")
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

i_abs_max = samples[value_col].abs().max()
i_abs_mean = samples[value_col].abs().mean()
print(f"|I|_max  = {i_abs_max*1e6:.3f} uA")
print(f"|I|_mean = {i_abs_mean*1e6:.3f} uA")

# 容忍阈值改成 10 µA, 因为 RSU+3m HRSU 引入约 1µA baseline 是正常的
if i_abs_max < 10e-6:
    print(f"✅ L1 PASS (|I|_max < 10 uA, baseline {i_abs_max*1e6:.2f} uA 含 RSU+电缆寄生)")
else:
    print(f"⚠️  |I|_max = {i_abs_max*1e6:.2f} uA, 高于 10µA 阈值, 检查:")
    print("    1) 探针/夹具有没有误接 DUT")
    print("    2) RSU 切换状态是不是真的开路")
    print("    3) 电缆屏蔽接地")


## 5. 落盘


In [ ]:
print("output paths:")
for k, v in result.paths.items():
    print(f"  {k:20s} -> {v}")
print(f"\nrun directory: {Path(list(result.paths.values())[0]).parent}")


---

## 通过标准回顾

- [ ] cell 1: dll 找到 + B1500 找到 + 通道列表正常
- [ ] cell 2-5: `complete == total`, `qc.status==ok`
- [ ] cell 6: 波形看到 5 个递减脉冲, 电流量级 ~1µA (RSU+电缆 baseline)
- [ ] cell 7: 路径打印正常

**通过 → yhzang 进 23 (wakeup L1)**

记录: 这台机器、这个 CHAN_ID、探针抬起 下的 baseline 电流是 \_\_\_ nA, 接器件后 DUT 电流需扣除此值。
